In [253]:
import pandas as pd
import numpy as np

In [254]:
extratos = pd.read_excel('extrato.xlsx')
botane = pd.read_excel('botane.xlsx')

In [255]:
datas_extrato = [f for f in extratos.columns if 'Data' in f or 'data' in f]
datas_botane = [f for f in botane.columns if 'Data' in f or 'data' in f]

for col in datas_extrato:
    extratos[col] = pd.to_datetime(extratos[col],dayfirst=True)
for col in datas_botane:
    botane[col] = pd.to_datetime(botane[col],dayfirst=True)



valores_limpos = extratos['Valor R$ '] \
    .str.replace('*', '', regex = False)\
    .str.replace('.', '', regex=False) \
    .str.replace(',', '.', regex=False) 


extratos['Valor R$ '] = pd.to_numeric(valores_limpos)


extratos.loc[extratos['Inf.'] == 'D', 'Valor R$ '] = extratos['Valor R$ '] * -1

In [256]:
botane_filtrado = botane[['Data de Competência','Data de Vencimento', 'Data pagamento/Recebimento', 'Tipo', 'Valor total pago/Recebido','Valor Juros', 'Banco']]
#botane_filtrado = botane_filtrado[botane_filtrado['Banco'] == '1 - BANCO DO BRASIL']


In [257]:
botane_filtrado['Data pagamento/Recebimento'] = botane_filtrado['Data pagamento/Recebimento'].dt.normalize()
botane_filtrado['Valor total pago/Recebido'] = botane_filtrado['Valor total pago/Recebido'].abs()


botane_filtrado.loc[botane_filtrado['Tipo'] == 'Despesa', 'Valor total pago/Recebido'] = \
    botane_filtrado['Valor total pago/Recebido'] * -1

C:\Users\arthu\AppData\Local\Temp\ipykernel_19896\1351403911.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  botane_filtrado['Data pagamento/Recebimento'] = botane_filtrado['Data pagamento/Recebimento'].dt.normalize()
C:\Users\arthu\AppData\Local\Temp\ipykernel_19896\1351403911.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  botane_filtrado['Valor total pago/Recebido'] = botane_filtrado['Valor total pago/Recebido'].abs()


In [258]:
extratos_filt = extratos[['Data balancete','Valor R$ ']]

botane_filtrado = botane_filtrado[
    botane_filtrado['Data pagamento/Recebimento'].isin(extratos_filt['Data balancete'])
]




In [281]:
soma_por_dia_botane_df = botane_filtrado.groupby(
    pd.Grouper(key='Data pagamento/Recebimento', freq='D')
)['Valor total pago/Recebido'].sum().reset_index()

soma_por_dia_botane_df  

soma_por_dia_extratos = extratos_filt.groupby(
    pd.Grouper(key='Data balancete', freq='D')
)['Valor R$ '].sum()


soma_por_dia_extratos_df = soma_por_dia_extratos.reset_index()
soma_por_dia_extratos_df

botane_sum_renamed = soma_por_dia_botane_df.rename(columns={
    'Data pagamento/Recebimento': 'Dia',
    'Valor total pago/Recebido': 'Valor Botane'
})


extratos_sum_renamed = soma_por_dia_extratos_df.rename(columns={
    'Data balancete': 'Dia',
    'Valor R$ ': 'Valor Extrato'  
})


tabela_comparativa = pd.merge(
    botane_sum_renamed,
    extratos_sum_renamed,
    on='Dia',
    how='inner'
)

botane_filtrado[botane_filtrado['Banco'] == '1 - BANCO DO BRASIL']
tabela_comparativa['Diferença'] = tabela_comparativa['Valor Botane'] - tabela_comparativa['Valor Extrato']


botane_filt = botane_filtrado[botane_filtrado['Data pagamento/Recebimento'] == '2025-08-18']
botane_filt = botane_filt[['Data pagamento/Recebimento', 'Valor total pago/Recebido']]
e = extratos_filt[extratos_filt['Data balancete'] == '2025-08-18']

print(botane_filt.describe())
print(e.describe())

botane_filtrado[botane_filtrado['Data pagamento/Recebimento'] == '2025-08-29']

      Data pagamento/Recebimento  Valor total pago/Recebido
count                         75                  75.000000
mean         2025-08-18 00:00:00                 232.854667
min          2025-08-18 00:00:00               -2328.020000
25%          2025-08-18 00:00:00                  53.000000
50%          2025-08-18 00:00:00                 195.000000
75%          2025-08-18 00:00:00                 520.000000
max          2025-08-18 00:00:00                7387.000000
std                          NaN                1102.757502
            Data balancete     Valor R$ 
count                   62  6.200000e+01
mean   2025-08-18 00:00:00  2.347083e-13
min    2025-08-18 00:00:00 -3.433413e+04
25%    2025-08-18 00:00:00 -5.668750e+01
50%    2025-08-18 00:00:00  2.047500e+02
75%    2025-08-18 00:00:00  5.525000e+02
max    2025-08-18 00:00:00  1.797938e+04
std                    NaN  5.132554e+03


,Data de Competência,Data de Vencimento,Data pagamento/Recebimento,Tipo,Valor total pago/Recebido,Valor Juros,Banco
18922,2025-07-03 16:43:00,2025-08-29 00:00:00,2025-08-29,Despesa,-2672.50,0.0,1 - BANCO DO BRASIL
18962,2025-07-04 18:30:00,2025-08-29 00:00:00,2025-08-29,Despesa,-1043.59,0.0,1 - BANCO DO BRASIL
19059,2025-07-14 17:07:00,2025-08-29 00:00:00,2025-08-29,Despesa,-460.67,0.0,1 - BANCO DO BRASIL
19163,2025-07-22 12:19:00,2025-08-29 00:00:00,2025-08-29,Despesa,-294.26,0.0,1 - BANCO DO BRASIL
19231,2025-07-31 16:10:00,2025-08-29 00:00:00,2025-08-29,Despesa,-2337.44,0.0,1 - BANCO DO BRASIL
...,...,...,...,...,...,...,...
28773,2025-08-29 14:19:29,2025-09-28 14:19:29,2025-08-29,Receita,3000.00,0.0,1 - BANCO DO BRASIL
28774,2025-08-29 14:22:55,2025-09-28 14:22:55,2025-08-29,Receita,550.00,0.0,1 - BANCO DO BRASIL
28775,2025-08-29 14:23:51,2025-09-28 14:23:51,2025-08-29,Receita,220.00,0.0,1 - BANCO DO BRASIL
28776,2025-08-29 14:24:07,2025-09-28 14:24:07,2025-08-29,Receita,460.00,0.0,1 - BANCO DO BRASIL
